# 03 - Search-Guided Self-Play and League Training

Learn policy targets from completed alpha-beta analyses and values from game outcomes. Collectors stay frozen during each iteration. CPU matches decide promotion.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Project and Dependencies

Keep Colab's existing CUDA PyTorch. Restart only if pip explicitly requires it.

In [ ]:
from pathlib import Path
import sys
import subprocess

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/Deep Reinforcement Learning/Chess")
if not (PROJECT_ROOT / "chess_rl").is_dir():
    raise FileNotFoundError(f"Project files are missing from {PROJECT_ROOT}. See README.md.")
sys.path.insert(0, str(PROJECT_ROOT))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-r", str(PROJECT_ROOT / "requirements_colab.txt")])

## Run Configuration

Edit configs/default.yaml once for the workflow. Use a new run_id for a changed experiment.

In [ ]:
from chess_rl.config import load_config, prepare_directories
from chess_rl.reproducibility import metadata, read_json, atomic_json, sha256

prepare_directories(PROJECT_ROOT)
cfg = load_config(PROJECT_ROOT)
print("Run:", cfg["run_id"])
print("Runtime:", metadata())

## Load Prepared Inputs

Notebook 02 must have completed. It does not need to remain open.

In [ ]:
initial = read_json(PROJECT_ROOT / "results" / cfg["run_id"] / "initial_selection.json")
initial_checkpoint = PROJECT_ROOT / initial["checkpoint"]
if sha256(initial_checkpoint) != initial["sha256"]:
    raise ValueError("Initialization checkpoint was modified")
dataset_manifest = read_json(PROJECT_ROOT / "datasets/manifests" / (cfg["run_id"] + ".json"))
print("Initialization:", initial_checkpoint)
print("Self-play settings:", cfg["self_play"])

## Targets and Exploration

The teacher searches every root action at a common completed depth with full windows. Target probabilities use softmax(score/0.25). Exploration changes the played move, not the target. Missing completed analyses have no policy target. Failed games do not become draw labels.

## Train or Resume the League

Default: 10 iterations, 512 games each, 2,000 updates. Saved games and update checkpoints survive disconnects. GPU collection follows the rule clock but does not measure CPU tournament speed.

In [ ]:
from chess_rl.self_play import run_league
champion = run_league(PROJECT_ROOT, cfg, initial_checkpoint, dataset_manifest)
print("Retained champion:", champion)

## Inspect Promotions

Promotion requires at least 55% score and a paired 95% interval above 50%, without operational failures. Rejected candidates and results are retained.

In [ ]:
from chess_rl.plots import plot_league
display(plot_league(PROJECT_ROOT, cfg["run_id"]))
league = read_json(PROJECT_ROOT / "checkpoints/self_play" / cfg["run_id"] / "league.json")
for item in league["iterations"]:
    print(item["iteration"], "promoted:", item["promoted"], "score:", item["summary"]["score"])

## Optional Self-Play or Search Study

Disabled by default. Trials initialize from the same checkpoint in separate folders. Search-only trials reuse weights; self-play trials train for three iterations.

In [ ]:
RUN_SEARCH_STUDY = False
SEARCH_ONLY = True
if RUN_SEARCH_STUDY:
    import yaml
    from chess_rl.tuning import run_self_play_search_study
    spaces = yaml.safe_load((PROJECT_ROOT / "configs/search_spaces.yaml").read_text())
    study_winner = run_self_play_search_study(PROJECT_ROOT, cfg, initial_checkpoint,
                                             dataset_manifest, spaces, search_only=SEARCH_ONLY)
    print("Research branch winner, not automatically final:", study_winner)
print("Open notebook 04 after all selected training runs finish.")